# H&M Dataset — FOAF Decentralized MF Experiment

Mirrors the ML-100K experiment structure exactly:
- Same FOAF gradient-sharing method (user-stratified, random-sampling batch)
- Same graph topologies: Random-2-out, Random-5-out, Scale-Free, Cycle
- Same evaluation metric: RMSE on held-out test set
- Same baselines: CL (centralized), FL (FedAvg), GL (Gossip), LL (local)

**Data source**: Download `articles.csv`, `customers.csv`, `transactions_train.csv`  
from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations

**Target**: purchase frequency `y_ij = count(customer_i, product_j)` — same as appendix.

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import datetime
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Dict, Optional
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import networkx as nx
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## 1. Data Loading & Preprocessing (follows H_M_Data.ipynb)

In [5]:
# ── Load raw files ──────────────────────────────────────────────────────────
DATA_DIR = "."   # change to path containing the three CSVs

df           = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                           dtype={"article_id": str})
customer_df  = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                           dtype={"article_id": str, "product_code": str})
article_df   = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                           dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

# ── Week index (0 = most recent week) ───────────────────────────────────────
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"]  = (df["t_dat"].max() - df["t_dat"]).dt.days // 7

print("Raw transactions:", len(df))

Raw transactions: 31788324


In [6]:
# ── Filter: top-6 product types, >=120-purchase customers, top-5000 locations
chosen_types = (
    article_df.groupby('product_type_name')['product_code']
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby('customer_id')['product_code'].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df['postal_code'].value_counts()[1:5001].index
customer_df = customer_df[customer_df['postal_code'].isin(location_customers)]

chosen_articles = article_df[article_df['product_type_name'].isin(chosen_types)]
df = df[df['product_code'].isin(chosen_articles['product_code'])]
df = df[df['customer_id'].isin(chosen_customers)]
df = df[df['customer_id'].isin(customer_df['customer_id'])]

df.drop(['t_dat', 'price', 'article_id', 'sales_channel_id'], axis=1, inplace=True)
df['bought'] = 1
df.reset_index(drop=True, inplace=True)

print(f'Filtered transactions: {len(df)} | Unique customers: {df["customer_id"].nunique()}')


Filtered transactions: 216390 | Unique customers: 1765


## 1b. Participant Selection & Train/Test Split

Participants are selected via **quantile-based deterministic sampling** before the train/test split:

1. Compute total interaction count $s_i$ for each user; discard users with $s_i < $ `MIN_INTERACTIONS`
2. Bin eligible users into `N_QUANTILES` equal-frequency bins by $s_i$
3. Allocate `TARGET_USERS` proportionally across bins; within each bin select the top users by $s_i$ (no random draws)
4. Random 75/25 train/test split on the selected rows (mirrors ML-100K)
5. Keep only test rows whose user appears in train; integer-encode IDs
6. Last 20% of train rows form the validation set for early stopping
7. Save to disk; reload on subsequent runs


In [8]:
# ── Configuration ────────────────────────────────────────────────────────────
TARGET_USERS     = 1760   # total users to select across all quantile bins
MIN_INTERACTIONS = 10    # activity floor: discard users below this threshold
N_QUANTILES      = 10     # number of equal-frequency quantile bins
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"


def select_by_quantile(df, target_users, min_interactions, n_quantiles=4,
                       user_col='customer_id', item_col='product_code'):
    """
    Quantile-based deterministic participant selection.

    Steps:
      1. Compute total interaction count s_i for each user across all time.
      2. Discard users with s_i < min_interactions (activity floor).
      3. Bin eligible users into n_quantiles equal-frequency bins based on s_i.
      4. Allocate target_users proportionally across bins (floor, then give
         remainder to highest bins to avoid under-selection).
      5. Within each bin, select the top-m_q users by s_i (highest activity).
         This is deterministic: no random draws, no seed required.

    Preserves the heterogeneous interaction-count distribution of the full
    dataset while guaranteeing every selected user has sufficient history.
    Returns the row-subset of df for the selected customers.
    """
    # Step 1: total interactions per user
    user_counts = (
        df.groupby(user_col)[item_col]
        .count()
        .rename('n_interactions')
        .reset_index()
    )

    # Step 2: activity floor
    eligible = user_counts[
        user_counts['n_interactions'] >= min_interactions
    ].copy().reset_index(drop=True)

    if len(eligible) == 0:
        raise ValueError(f"No users with >= {min_interactions} interactions.")

    # Step 3: quantile binning on eligible users
    # pd.qcut produces equal-frequency bins; duplicates='drop' merges ties
    eligible['quantile_bin'] = pd.qcut(
        eligible['n_interactions'], q=n_quantiles,
        labels=False, duplicates='drop'
    )
    actual_bins = sorted(eligible['quantile_bin'].unique())
    n_actual    = len(actual_bins)

    # Step 4: proportional allocation of target_users across bins
    total_eligible = len(eligible)
    bin_sizes = eligible.groupby('quantile_bin').size()

    # Base allocation (floor)
    alloc = {
        b: max(1, int(target_users * bin_sizes[b] / total_eligible))
        for b in actual_bins
    }
    # Distribute any remainder to the highest bins (most active users first)
    remainder = target_users - sum(alloc.values())
    for b in sorted(actual_bins, reverse=True):
        if remainder <= 0:
            break
        cap = int(bin_sizes[b])
        add = min(remainder, cap - alloc[b])
        alloc[b] += add
        remainder -= add

    # Step 5: within each bin, pick the top-m_q users by n_interactions
    selected_users = []
    for b in actual_bins:
        grp = (
            eligible[eligible['quantile_bin'] == b]
            .sort_values('n_interactions', ascending=False)
        )
        m_q = min(alloc[b], len(grp))
        selected_users.append(grp.head(m_q))

    selected_df   = pd.concat(selected_users, ignore_index=True)
    selected_ids  = selected_df[user_col].values

    # Summary
    print(f"  Eligible users (>={min_interactions} interactions): {total_eligible}")
    print(f"  Quantile bins: {n_actual}  |  Target: {target_users}")
    for b in actual_bins:
        grp = eligible[eligible['quantile_bin'] == b]['n_interactions']
        m_q = min(alloc[b], int(bin_sizes[b]))
        print(f"    Bin {b}: range [{grp.min()}, {grp.max()}]  "
              f"eligible={int(bin_sizes[b])}  selected={m_q}")
    print(f"  Total selected: {len(selected_ids)} users")

    df_sub = df[df[user_col].isin(selected_ids)].copy()
    df_sub.reset_index(drop=True, inplace=True)
    return df_sub


# ── Save-or-load pattern ──────────────────────────────────────────────────────
cache_tag  = f'u{TARGET_USERS}_m{MIN_INTERACTIONS}_q{N_QUANTILES}'
train_path = os.path.join(SAMPLED_DATA_DIR, f'train_{cache_tag}.csv')
val_path   = os.path.join(SAMPLED_DATA_DIR, f'val_{cache_tag}.csv')
test_path  = os.path.join(SAMPLED_DATA_DIR, f'test_{cache_tag}.csv')
meta_path  = os.path.join(SAMPLED_DATA_DIR, f'meta_{cache_tag}.csv')

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload ────────────────────────────────────────────────────
    print(f'Loading cached dataset ({cache_tag}) from {SAMPLED_DATA_DIR}/')
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df['key'] == 'n_users', 'value'].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df['key'] == 'n_items', 'value'].iloc[0])
    print(f'  Loaded: {N_USERS} users | {N_ITEMS} items')
    print(f'  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

else:
    # ── Slow path ────────────────────────────────────────────────────────────
    print('Cache not found — running pipeline...')

    # Step 1: quantile-based selection on full filtered rows
    df_sampled = select_by_quantile(
        df,
        target_users=TARGET_USERS,
        min_interactions=MIN_INTERACTIONS,
        n_quantiles=N_QUANTILES,
    )

    # Step 2: random 75/25 train/test split (mirrors ML-100K)
    X = df_sampled[['customer_id', 'product_code']].values
    y = df_sampled['bought'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED
    )

    train_df = pd.DataFrame(X_train, columns=['customer_id', 'product_code'])
    test_df  = pd.DataFrame(X_test,  columns=['customer_id', 'product_code'])
    train_df['bought'] = y_train
    test_df['bought']  = y_test

    # Step 3: keep only test rows whose user appears in train
    train_users = set(train_df['customer_id'])
    mask        = test_df['customer_id'].isin(train_users)
    test_df     = test_df[mask].copy()
    print(f'  Dropped {(~mask).sum()} test rows with users absent from train')

    # Step 4: integer encoding — fit on combined train+test (mirrors ML-100K)
    all_users = np.unique(
        pd.concat([train_df[['customer_id']], test_df[['customer_id']]])['customer_id']
    )
    all_items = np.unique(
        pd.concat([train_df[['product_code']], test_df[['product_code']]])['product_code']
    )
    customer_id2idx = {c: i for i, c in enumerate(all_users)}
    item_id2idx     = {a: i for i, a in enumerate(all_items)}
    for df_ in [train_df, test_df]:
        df_['customer_id']  = df_['customer_id'].map(customer_id2idx)
        df_['product_code'] = df_['product_code'].map(item_id2idx)

    N_USERS = len(all_users)
    N_ITEMS = len(all_items)
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # Step 5: val split — last 20% of train rows
    n_val    = int(len(train_df) * 0.2)
    val_df   = train_df.tail(n_val).reset_index(drop=True)
    train_df = train_df.head(len(train_df) - n_val).reset_index(drop=True)

    # Step 6: save
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    pd.DataFrame([
        {'key': 'n_users',          'value': N_USERS},
        {'key': 'n_items',          'value': N_ITEMS},
        {'key': 'n_train',          'value': len(train_df)},
        {'key': 'n_test',           'value': len(test_df)},
        {'key': 'target_users',     'value': TARGET_USERS},
        {'key': 'min_interactions', 'value': MIN_INTERACTIONS},
        {'key': 'n_quantiles',      'value': N_QUANTILES},
        {'key': 'selection',        'value': 'quantile_top_k'},
        {'key': 'split',            'value': 'random_75_25'},
    ]).to_csv(meta_path, index=False)

    print(f'  Saved to {SAMPLED_DATA_DIR}/')
    print(f'  Users   : {N_USERS}')
    print(f'  Items   : {N_ITEMS}')
    print(f'  Train   : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
    print(f'  Density : {len(train_df)/(N_USERS*N_ITEMS)*100:.4f}%')


Cache not found — running pipeline...
  Eligible users (>=10 interactions): 1765
  Quantile bins: 10  |  Target: 1760
    Bin 0: range [32, 69]  eligible=179  selected=178
    Bin 1: range [70, 81]  eligible=193  selected=192
    Bin 2: range [82, 90]  eligible=168  selected=167
    Bin 3: range [91, 100]  eligible=179  selected=178
    Bin 4: range [101, 110]  eligible=176  selected=175
    Bin 5: range [111, 121]  eligible=169  selected=169
    Bin 6: range [122, 135]  eligible=172  selected=172
    Bin 7: range [136, 154]  eligible=183  selected=183
    Bin 8: range [155, 188]  eligible=169  selected=169
    Bin 9: range [189, 528]  eligible=177  selected=177
  Total selected: 1760 users
  Dropped 0 test rows with users absent from train
  Saved to /Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled/
  Users   : 1760
  Items   : 12777
  Train   : 129608 | Val: 32402 | Test: 54004
  Density : 0.5764%


## 2. Dataset & DataLoader

In [10]:
class UserRandomSamplingDataset(Dataset):
    """
    Per-user dataset: each __getitem__ returns a random batch of that user's
    interactions.  Mirrors the ML-100K DataLoader design.
    """
    def __init__(self, df: pd.DataFrame, batch_size: int = 10):
        self.batch_size = batch_size
        # Build user -> indices lookup once
        users = torch.tensor(df["customer_id"].values, dtype=torch.long)
        items = torch.tensor(df["product_code"].values, dtype=torch.long)
        targets = torch.tensor(df["bought"].values, dtype=torch.float32)

        unique_users = users.unique()
        self.user_ids = unique_users
        self.user_to_indices = {
            u.item(): (users == u).nonzero(as_tuple=True)[0]
            for u in unique_users
        }
        self.items   = items
        self.targets = targets

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid = self.user_ids[idx].item()
        indices = self.user_to_indices[uid]
        bs = min(len(indices), self.batch_size)
        sel = indices[torch.randperm(len(indices))[:bs]]
        # Return (user_idx, item_idx) pairs and targets
        x = torch.stack([
            torch.full((bs,), uid, dtype=torch.long),
            self.items[sel]
        ], dim=1)  # shape: (bs, 2)  col0=user, col1=item
        y = self.targets[sel]  # shape: (bs,)
        return x, y


BATCH_SIZE = 10

train_dataset = UserRandomSamplingDataset(train_df, batch_size=BATCH_SIZE)
val_dataset   = UserRandomSamplingDataset(val_df,   batch_size=BATCH_SIZE)
test_dataset  = UserRandomSamplingDataset(test_df,  batch_size=BATCH_SIZE)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

print(f"Train loader batches: {len(train_loader)} (= N_USERS = {N_USERS})")

Train loader batches: 1760 (= N_USERS = 1760)
